## <center> **Medical Insurance Cost Prediction** </center>
---

### <center> **Data Loading and Previewing** </center>

### Imports

In [121]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import joblib
import json

### Load the data

In [51]:
df = pd.read_csv("insurance.csv")

### First five rows

In [52]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19.0,female,27.900,0,yes,southwest,16884.92400
1,18.0,male,33.770,1,no,southeast,1725.55230
2,28.0,male,33.000,3,no,southeast,4449.46200
3,33.0,male,22.705,0,no,northwest,21984.47061
4,32.0,male,28.880,0,no,northwest,3866.85520


### Last five rows

In [53]:
df.tail()

,age,sex,bmi,children,smoker,region,charges
1333,50.0,male,30.97,3,no,northwest,10600.5483
1334,18.0,female,31.92,0,no,northeast,2205.9808
1335,18.0,female,36.85,0,no,southeast,1629.8335
1336,21.0,female,25.80,0,no,southwest,2007.9450
1337,61.0,female,29.07,0,yes,northwest,29141.3603


### Random five rows

In [54]:
df.sample(5)

,age,sex,bmi,children,smoker,region,charges
851,61.0,male,32.300,2,no,northwest,14119.62000
451,30.0,male,24.130,1,no,northwest,4032.24070
986,43.0,male,30.115,3,no,northwest,8410.04685
1146,60.0,male,32.800,0,yes,southwest,52590.82939
77,21.0,male,35.530,0,no,southeast,1532.46970


### Number of rows and columns

In [55]:
df.shape

(1338, 7)

### Categorical columns

In [ ]:
categorical_cols = df.select_dtypes(include=["object"])
for column in categorical_cols:
    print(f"- {column}")

### Numeric columns

In [ ]:
numeric_cols = df.select_dtypes(include=["number"])

for column in numeric_cols:
    print(f"- {column}")

### Columns, datatypes and Indexrange

In [56]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1336 non-null   float64
 1   sex       1337 non-null   object 
 2   bmi       1336 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1337 non-null   object 
 5   region    1336 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(3), int64(1), object(3)
memory usage: 73.3+ KB


### Statistical summary

In [57]:
df.describe()

,age,bmi,children,charges
count,1336.000000,1336.000000,1338.000000,1338.000000
mean,39.225299,30.653447,1.094918,13270.422265
std,14.048210,6.096546,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.272500,0.000000,4740.287150
50%,39.000000,30.380000,1.000000,9382.033000
75%,51.000000,34.618750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


### Null values

In [58]:
df.isnull().sum()

age         2
sex         1
bmi         2
children    0
smoker      1
region      2
charges     0
dtype: int64

### <center> **Data Visualization** <center>
---

### Charges based on Sex

In [ ]:
sns.barplot(
    x= df["sex"],
    y= df["charges"],
    width=0.7
)

plt.show()

### Charges based on Smoking status

In [ ]:
sns.barplot(
    x= df["smoker"],
    y= df["charges"],
    width=0.7
)

plt.show()

### Percentage of charges based on Gender

In [ ]:
charges_percentage_gender = df.groupby("sex")["charges"].sum()

plt.pie(
    x= charges_percentage_gender.values,
    labels= charges_percentage_gender.index,
    autopct="%.2f%%"
)

plt.show()

### Charges based on Region

In [ ]:
sns.barplot(
    x= df["region"],
    y= df["charges"],
    palette="viridis"
)

plt.show()

### Percentage of charges based on Smoking status

In [ ]:
charges_percentage_smoking = df.groupby("smoker")["charges"].sum()

plt.pie(
    x= charges_percentage_smoking.values,
    labels= charges_percentage_smoking.index,
    autopct="%.2f%%"
)
plt.show()

### Charges based on Children

In [ ]:
sns.lineplot(
    x= df["children"],
    y= df["charges"],
    color="purple"
)

plt.show()

### Percentage of charges based on Region

In [ ]:
charges_percentage_region = df.groupby("region")["charges"].sum()
plt.pie(
    x= charges_percentage_region.values,
    labels= charges_percentage_region.index,
    autopct="%.2f%%"
)

plt.show()

### Charges based on BMI

In [ ]:
sns.lineplot(
    x= df["charges"],
    y= df["bmi"],
    color="blue"
)

plt.show()

### Charges based on Age

In [ ]:
sns.lineplot(
    x= df["age"],
    y= df["charges"],
    color="yellow"
)
plt.show()

### <center> **Data Preprocessing and Model Training** </center>
---

### Features and Target

In [59]:
X = df.drop("charges", axis=1)
y = df["charges"]

### Training and Test sets

In [77]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [61]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19.0,female,27.900,0,yes,southwest,16884.92400
1,18.0,male,33.770,1,no,southeast,1725.55230
2,28.0,male,33.000,3,no,southeast,4449.46200
3,33.0,male,22.705,0,no,northwest,21984.47061
4,32.0,male,28.880,0,no,northwest,3866.85520


### Separate Categorical and Numeric columns

In [78]:
numerical_features = [
    "age",
    "bmi",
    "children",
]

categorical_features = [
    "sex",
    "smoker",
    "region"
]

### Build preprocessing pipeline

In [79]:
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)

categorical_transformer = Pipeline(
    steps= [
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

### Column Transformer

In [80]:
preprocessor = ColumnTransformer(
    transformers= [
        (
            "num",
            numeric_transformer,
            numerical_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

### Create Decision Tree pipeline

In [108]:
pipeline = Pipeline(
    steps= [
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            RandomForestRegressor(
                random_state=42
            )
        )
    ]
)


### Train Model

In [109]:
model = pipeline.fit(X_train, y_train)

### Make Predictions

In [110]:
y_test_pred = model.predict(X_test)
y_train_pred = model.predict(X_train)
print(y_test_pred)

[ 9635.226465   6030.1933399 28325.8993115 12372.0766665 34740.4070437
  8336.2078785  2136.871807  14715.8631332  6061.0045165 11053.7967965
 19181.6831886  6910.9485555  4487.5573374 46270.3461023 48699.8381584
 45235.8860253 10573.67084   43612.5430069 10221.2925575 24878.1053248
  6161.0907474  8742.6061206  1539.320941   2625.1003675 12583.0844128
 12773.5121524 14599.8965965  5164.9636894 10648.4522581  3802.0210271
  7697.499577  11716.3700655  2973.350004   6246.4387752  3590.3633826
 11986.4843113  4304.5254491  7866.2570474 23885.5140495  5580.7406058
  7277.2255223  2726.5197003 14969.7173226 13864.1641362  6280.496138
 16154.3975256 18206.4702394  7204.5158697 41858.7112164  6416.5069549
 13978.466564   2267.475323   6351.8253884  2029.0496134 11584.8541422
 11135.4193488  3740.3440642 46055.8064045 11876.7701435 15543.5649466
 13789.1457305  7662.1073234 21401.4130523  7395.7813065 12885.1219812
  6334.2436594 18044.0645418 12868.974878   6738.4720346  1978.6844525
  7102.

### Campare results

In [111]:
# Combine actual and predicted values side by side
results = np.column_stack((y_test, y_test_pred))

# Printing the results
print("Actual Values  |  Predicted Values")
print("-----------------------------")
for actual, predicted in results:
    print(f"{actual:14.2f} |  {predicted:12.2f}")

Actual Values  |  Predicted Values
-----------------------------
       9095.07 |       9635.23
       5272.18 |       6030.19
      29330.98 |      28325.90
       9301.89 |      12372.08
      33750.29 |      34740.41
       4536.26 |       8336.21
       2117.34 |       2136.87
      14210.54 |      14715.86
       3732.63 |       6061.00
      10264.44 |      11053.80
      18259.22 |      19181.68
       7256.72 |       6910.95
       3947.41 |       4487.56
      46151.12 |      46270.35
      48673.56 |      48699.84
      44202.65 |      45235.89
       9800.89 |      10573.67
      42969.85 |      43612.54
       8233.10 |      10221.29
      21774.32 |      24878.11
       5080.10 |       6161.09
       7441.50 |       8742.61
       1256.30 |       1539.32
       2755.02 |       2625.10
      11085.59 |      12583.08
      10923.93 |      12773.51
      12644.59 |      14599.90
      18804.75 |       5164.96
       9715.84 |      10648.45
       1131.51 |       3802.02
     

### Evalaute results

In [112]:
train_mse = mean_squared_error(y_train_pred, y_train)
test_mse = mean_squared_error(y_test_pred, y_test)

train_mae = mean_absolute_error(y_train_pred, y_train)
test_mae = mean_absolute_error(y_test_pred, y_test)

train_r2 = r2_score(y_train_pred, y_train)
test_r2 = r2_score(y_test_pred, y_test)

print(f"\nTrain MSE: {train_mae}")
print(f"Test MSE: {test_mae}\n")

print(f"\nTrain MAE: {train_mae}")
print(f"Test MAE: {test_mae}\n")

print(f"\nTrain R2: {train_r2}")
print(f"Test R2: {test_r2}")


Train MSE: 1043.701329566881
Test MSE: 2666.369257433831


Train MAE: 1043.701329566881
Test MAE: 2666.369257433831


Train R2: 0.9748431882491034
Test R2: 0.8243777254102329


### Save Model

In [122]:
joblib.dump(model, "delivery_time_model.pkl")

metadata = {
    "categorical_options": {
        col: sorted(df[col].dropna().unique().tolist())
        for col in ['sex', 'smoker', 'region']
    },
    "numeric_ranges": {
        "age": [float(df["age"].min()), float(df["age"].max())],
        "bmi": [float(df["bmi"].min()), float(df["bmi"].max())],
        "children": [float(df["children"].min()), float(df["children"].max())],
    }
}

with open("model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)